In [1]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REQUIRED_DATA_FILES = {
    "train_demos.pkl",
    "valid_scenarios.pkl",
    "test_scenarios.pkl",
}


def contains_required_files(folder: Path) -> bool:
    return folder.is_dir() and REQUIRED_DATA_FILES.issubset(
        {path.name for path in folder.iterdir() if path.is_file()}
    )


def find_data_directory() -> Path | None:
    candidates = [
        Path("/kaggle/input/competitions/ioai-2026-home-task-2"),
        Path("/kaggle/input/ioai-2026-home-task-2"),
        Path("/content/data"),
        Path("data"),
    ]

    for candidate in candidates:
        if contains_required_files(candidate):
            return candidate

    # Kaggle puede montar la competencia bajo una carpeta con nombre ligeramente distinto.
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.is_dir():
        for train_file in kaggle_root.rglob("train_demos.pkl"):
            if contains_required_files(train_file.parent):
                return train_file.parent

    return None


DATA_DIR = find_data_directory()

if DATA_DIR is None:
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])

    import gdown

    download_dir = Path("data")
    download_dir.mkdir(parents=True, exist_ok=True)

    print("No se encontraron los datos localmente. Descargando la carpeta pública...")
    gdown.download_folder(
        id="1DXFDoY9bqulMBFacyDShVIx8Sa7Z5Wpa",
        output=str(download_dir),
        quiet=False,
        use_cookies=False,
    )
    DATA_DIR = find_data_directory()

if DATA_DIR is None:
    raise FileNotFoundError(
        "No fue posible encontrar train_demos.pkl, valid_scenarios.pkl y "
        "test_scenarios.pkl. En Kaggle, agregue los datos de la competencia "
        "al notebook; en Colab, habilite internet para la descarga pública."
    )

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("Archivos:", sorted(path.name for path in DATA_DIR.iterdir()))

DATA_DIR: /kaggle/input/competitions/ioai-2026-home-task-2
OUTPUT_DIR: /kaggle/working
Archivos: ['test_scenarios.pkl', 'train_demos.pkl', 'valid_scenarios.pkl']


In [2]:
import json
import pickle
import random
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from IPython.display import Image as NotebookImage, display
from PIL import Image as PILImage, ImageDraw
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GRID_SIZE = 8
N_DEPOTS = 6
MAX_STEPS = 120
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ACTION_NAMES = {
    0: "south",
    1: "north",
    2: "east",
    3: "west",
    4: "pickup",
    5: "dropoff",
}

ACTION_DELTAS = {
    0: (1, 0),
    1: (-1, 0),
    2: (0, 1),
    3: (0, -1),
}

DEPOT_NAMES = ["A", "B", "C", "D", "E", "F"]

print("data dir:", DATA_DIR)
print("device:", DEVICE)

data dir: /kaggle/input/competitions/ioai-2026-home-task-2
device: cpu


In [3]:
class DeliverySimulator8x8:
    """Run one 8x8 delivery episode."""

    def reset(self, scenario: dict[str, Any]) -> tuple[int, int, int, int]:
        """Start a scenario and return the compact state."""
        self.step_count = 0
        self.carrying = False
        self.walls = {tuple(cell) for cell in scenario["walls"]}
        self.depots = [tuple(cell) for cell in scenario["depots"]]
        self.agent_pos = tuple(scenario["agent_pos"])
        self.package_location = int(scenario["package_location"])
        self.destination = int(scenario["destination"])
        return self.state()

    def state(self) -> tuple[int, int, int, int]:
        """Return row, column, package field, and destination."""
        package_field = N_DEPOTS if self.carrying else self.package_location
        return int(self.agent_pos[0]), int(self.agent_pos[1]), int(package_field), int(self.destination)

    def can_enter(self, row: int, col: int) -> bool:
        """Check whether the robot can occupy a cell."""
        return 0 <= row < GRID_SIZE and 0 <= col < GRID_SIZE and (row, col) not in self.walls

    def valid_action_mask(self) -> np.ndarray:
        """Return the currently valid actions."""
        row, col, _, destination = self.state()
        mask = np.zeros(6, dtype=bool)
        for action, (dr, dc) in ACTION_DELTAS.items():
            mask[action] = self.can_enter(row + dr, col + dc)
        mask[4] = (not self.carrying) and self.agent_pos == self.depots[self.package_location]
        mask[5] = self.carrying and self.agent_pos == self.depots[destination]
        return mask

    def observation(self) -> dict[str, Any]:
        """Build the model observation for the current state."""
        row, col, package_field, destination = self.state()
        carrying = package_field == N_DEPOTS
        dest_row, dest_col = self.depots[destination]
        target_row, target_col = (dest_row, dest_col) if carrying else self.depots[package_field]

        grid = np.zeros((6, GRID_SIZE, GRID_SIZE), dtype=np.float32)
        for wr, wc in self.walls:
            grid[0, wr, wc] = 1.0
        for dr, dc in self.depots:
            grid[1, dr, dc] = 1.0
        grid[2, row, col] = 1.0
        if not carrying:
            pr, pc = self.depots[package_field]
            grid[3, pr, pc] = 1.0
        grid[4, dest_row, dest_col] = 1.0
        grid[5, :, :] = float(carrying)

        blocked_moves = [float(not self.can_enter(row + dr, col + dc)) for dr, dc in ACTION_DELTAS.values()]
        vector = np.array(
            [
                row / (GRID_SIZE - 1),
                col / (GRID_SIZE - 1),
                package_field / N_DEPOTS,
                destination / (N_DEPOTS - 1),
                float(carrying),
                target_row / (GRID_SIZE - 1),
                target_col / (GRID_SIZE - 1),
                (target_row - row) / (GRID_SIZE - 1),
                (target_col - col) / (GRID_SIZE - 1),
                *blocked_moves,
            ],
            dtype=np.float32,
        )
        return {"grid": grid, "vector": vector, "action_mask": self.valid_action_mask(), "state": self.state()}

    def step(self, action: int) -> tuple[tuple[int, int, int, int], bool, bool, dict[str, Any]]:
        """Apply one action and report episode status."""
        action = int(action)
        done = False
        info = {"invalid_pickup_or_dropoff": False}

        if action in ACTION_DELTAS:
            dr, dc = ACTION_DELTAS[action]
            row, col = self.agent_pos[0] + dr, self.agent_pos[1] + dc
            if self.can_enter(row, col):
                self.agent_pos = (row, col)
        elif action == 4 and (not self.carrying) and self.agent_pos == self.depots[self.package_location]:
            self.carrying = True
        elif action == 5 and self.carrying and self.agent_pos == self.depots[self.destination]:
            done = True
            self.carrying = False
            self.package_location = self.destination
        elif action in (4, 5):
            info["invalid_pickup_or_dropoff"] = True
        else:
            raise ValueError(f"unknown action: {action}")

        self.step_count += 1
        return self.state(), done, self.step_count >= MAX_STEPS and not done, info

    def render(self) -> str:
        """Return an ASCII rendering of the current grid."""
        grid = [["." for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        for row, col in self.walls:
            grid[row][col] = "#"
        for i, (row, col) in enumerate(self.depots):
            grid[row][col] = DEPOT_NAMES[i]

        agent_row, agent_col = self.agent_pos
        grid[agent_row][agent_col] = "T*" if self.carrying else "T"
        rows = [" ".join(f"{cell:>2}" for cell in row) for row in grid]
        package_name = "in taxi" if self.carrying else DEPOT_NAMES[self.package_location]
        rows.append(f"package={package_name}, destination={DEPOT_NAMES[self.destination]}")
        return "\n".join(rows)

In [4]:
with (DATA_DIR / "train_demos.pkl").open("rb") as f:
    train_data = pickle.load(f)
with (DATA_DIR / "valid_scenarios.pkl").open("rb") as f:
    valid_scenarios = pickle.load(f)
with (DATA_DIR / "test_scenarios.pkl").open("rb") as f:
    test_scenarios = pickle.load(f)

train_trajectories = train_data["trajectories"]
steps = [t["num_steps"] for t in train_trajectories]

print("Loaded data")
print("  training demonstrations:", len(train_trajectories))
print("  validation scenarios:", len(valid_scenarios))
print("  test scenarios:", len(test_scenarios))
print("  training state-action samples:", sum(steps))
print("  average demonstration length:", f"{np.mean(steps):.2f}")
print("  expert success rate:", f"{100 * np.mean([t['success'] for t in train_trajectories]):.1f}%")


Loaded data
  training demonstrations: 400
  validation scenarios: 200
  test scenarios: 1600
  training state-action samples: 5327
  average demonstration length: 13.32
  expert success rate: 100.0%


In [5]:
def add_target_channel(obs):
    """
    Adds a 7th channel to the grid.
    The new channel marks the current target cell:
    - before pickup: package location
    - after pickup: destination location
    """

    grid = obs["grid"].astype(np.float32)  # shape: (6, 8, 8)
    vector = obs["vector"].astype(np.float32)

    # In the notebook vector:
    # vector[5] = target_row normalized
    # vector[6] = target_col normalized
    target_row = int(round(vector[5] * (GRID_SIZE - 1)))
    target_col = int(round(vector[6] * (GRID_SIZE - 1)))

    target_channel = np.zeros((1, GRID_SIZE, GRID_SIZE), dtype=np.float32)
    target_channel[0, target_row, target_col] = 1.0

    enhanced_grid = np.concatenate([grid, target_channel], axis=0)

    return enhanced_grid

In [6]:
START_ACTION = 6
PREVIOUS_ACTION_VOCAB = 7


class ModularNavigationDataset(Dataset):
    """
    Mantiene la información del paso anterior de cada trayectoria.

    Cada muestra contiene:
        - cuadrícula actual
        - vector actual
        - acción anterior
        - máscara de acciones válidas
        - acción experta actual
    """

    def __init__(self, trajectories):
        self.samples = []

        for trajectory in trajectories:
            previous_action = START_ACTION

            for obs, action in zip(
                trajectory["observations"],
                trajectory["actions"],
                strict=True
            ):
                self.samples.append({
                    "observation": obs,
                    "previous_action": previous_action,
                    "action": int(action),
                })

                previous_action = int(action)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]

        obs = sample["observation"]
        previous_action = sample["previous_action"]
        action = sample["action"]

        grid = torch.tensor(
            add_target_channel(obs),
            dtype=torch.float32
        )

        vector = torch.tensor(
            obs["vector"],
            dtype=torch.float32
        )

        previous_action = torch.tensor(
            previous_action,
            dtype=torch.long
        )

        mask = torch.tensor(
            obs["action_mask"],
            dtype=torch.bool
        )

        action = torch.tensor(
            action,
            dtype=torch.long
        )

        return grid, vector, previous_action, mask, action

In [7]:
modular_train_dataset = ModularNavigationDataset(
    train_trajectories
)

modular_train_loader = DataLoader(
    modular_train_dataset,
    batch_size=128,
    shuffle=True
)

grid_example, vector_example, previous_example, mask_example, y_example = (
    modular_train_dataset[0]
)

print("Samples:", len(modular_train_dataset))
print("Grid shape:", grid_example.shape)
print("Vector shape:", vector_example.shape)
print("Previous action:", int(previous_example))
print("Target action:", int(y_example))

Samples: 5327
Grid shape: torch.Size([7, 8, 8])
Vector shape: torch.Size([13])
Previous action: 6
Target action: 1


In [8]:
class ModularNavigationModel(nn.Module):
    """
    Primer módulo:
        CNN que analiza espacialmente la cuadrícula.

    Segundo módulo:
        Cabeza de decisión que combina:
        - característica de la posición del robot
        - característica del objetivo
        - características de las cuatro celdas vecinas
        - resumen global del tablero
        - vector numérico
        - acción anterior
    """

    def __init__(
        self,
        vector_dim=13,
        num_actions=6,
        previous_action_vocab=7
    ):
        super().__init__()

        # No se aplana el mapa aquí.
        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(
                7,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                96,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(96),
            nn.ReLU()
        )

        # Procesamiento de las 13 características globales.
        self.vector_encoder = nn.Sequential(
            nn.Linear(vector_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 64),
            nn.ReLU()
        )

        # Representación aprendida de la acción anterior.
        self.previous_action_embedding = nn.Embedding(
            previous_action_vocab,
            16
        )

        # Se extraen siete grupos espaciales:
        # robot + objetivo + cuatro vecinos + resumen global.
        spatial_decision_dim = 96 * 7

        decision_input_dim = (
            spatial_decision_dim
            + 64   # vector
            + 16   # acción anterior
        )

        self.decision_head = nn.Sequential(
            nn.Linear(decision_input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(128, num_actions)
        )

    @staticmethod
    def gather_position_features(
        feature_map,
        rows,
        columns
    ):
        """
        Extrae el vector de características de una posición
        para cada elemento del batch.
        """
        batch_indices = torch.arange(
            feature_map.size(0),
            device=feature_map.device
        )

        return feature_map[
            batch_indices,
            :,
            rows,
            columns
        ]

    def forward(
        self,
        grid,
        vector,
        previous_action
    ):
        """
        grid:
            [batch, 7, 8, 8]

        vector:
            [batch, 13]

        previous_action:
            [batch]
        """

        feature_map = self.spatial_encoder(grid)

        height = feature_map.size(2)
        width = feature_map.size(3)

        # Posición del robot:
        # vector[0] = row normalizada
        # vector[1] = col normalizada
        robot_rows = torch.round(
            vector[:, 0] * (height - 1)
        ).long()

        robot_columns = torch.round(
            vector[:, 1] * (width - 1)
        ).long()

        robot_rows = robot_rows.clamp(
            min=0,
            max=height - 1
        )

        robot_columns = robot_columns.clamp(
            min=0,
            max=width - 1
        )

        # Posición del objetivo:
        # vector[5] = target row normalizada
        # vector[6] = target col normalizada
        target_rows = torch.round(
            vector[:, 5] * (height - 1)
        ).long()

        target_columns = torch.round(
            vector[:, 6] * (width - 1)
        ).long()

        target_rows = target_rows.clamp(
            min=0,
            max=height - 1
        )

        target_columns = target_columns.clamp(
            min=0,
            max=width - 1
        )

        # Características exactamente donde está el robot.
        robot_features = self.gather_position_features(
            feature_map,
            robot_rows,
            robot_columns
        )

        # Características exactamente donde está el objetivo.
        target_features = self.gather_position_features(
            feature_map,
            target_rows,
            target_columns
        )

        # Extraer las cuatro celdas vecinas en el mismo orden
        # de ACTION_DELTAS:
        # south, north, east, west.
        neighbor_features = []

        for action_id in range(4):
            dr, dc = ACTION_DELTAS[action_id]

            neighbor_rows = robot_rows + dr
            neighbor_columns = robot_columns + dc

            valid_position = (
                (neighbor_rows >= 0)
                & (neighbor_rows < height)
                & (neighbor_columns >= 0)
                & (neighbor_columns < width)
            )

            safe_rows = neighbor_rows.clamp(
                min=0,
                max=height - 1
            )

            safe_columns = neighbor_columns.clamp(
                min=0,
                max=width - 1
            )

            current_neighbor_features = (
                self.gather_position_features(
                    feature_map,
                    safe_rows,
                    safe_columns
                )
            )

            # Una posición fuera del tablero se representa con ceros.
            current_neighbor_features = (
                current_neighbor_features
                * valid_position.unsqueeze(1).to(
                    feature_map.dtype
                )
            )

            neighbor_features.append(
                current_neighbor_features
            )

        neighbor_features = torch.cat(
            neighbor_features,
            dim=1
        )

        # Resumen global aprendido del mapa.
        global_features = feature_map.mean(
            dim=(2, 3)
        )

        vector_features = self.vector_encoder(
            vector
        )

        previous_action_features = (
            self.previous_action_embedding(
                previous_action
            )
        )

        decision_features = torch.cat(
            [
                robot_features,
                target_features,
                neighbor_features,
                global_features,
                vector_features,
                previous_action_features
            ],
            dim=1
        )

        return self.decision_head(
            decision_features
        )

In [9]:
modular_model = ModularNavigationModel(
    vector_dim=vector_example.numel(),
    num_actions=6
).to(DEVICE)

print(modular_model)

total_parameters = sum(
    parameter.numel()
    for parameter in modular_model.parameters()
)

print("Parameters:", total_parameters)

ModularNavigationModel(
  (spatial_encoder): Sequential(
    (0): Conv2d(7, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): ReLU()
  )
  (vector_encoder): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): R

In [10]:
modular_action_counts = Counter(
    sample["action"]
    for sample in modular_train_dataset.samples
)

modular_counts = torch.zeros(
    6,
    dtype=torch.float32
)

for action_id, count in modular_action_counts.items():
    modular_counts[action_id] = count

modular_class_weights = (
    modular_counts.sum()
    / (
        6
        * modular_counts.clamp(min=1)
    )
)

modular_class_weights = (
    modular_class_weights
    / modular_class_weights.mean()
)

modular_class_weights = modular_class_weights.to(
    DEVICE
)

print("Action counts:", modular_action_counts)
print("Class weights:", modular_class_weights)

Action counts: Counter({3: 1193, 1: 1148, 2: 1120, 0: 1066, 4: 400, 5: 400})
Class weights: tensor([0.6591, 0.6120, 0.6273, 0.5889, 1.7564, 1.7564])


In [11]:
modular_optimizer = torch.optim.AdamW(
    modular_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

modular_criterion = nn.CrossEntropyLoss(
    weight=modular_class_weights
)

In [12]:
@torch.no_grad()
def modular_train_accuracy():
    modular_model.eval()

    correct = 0
    total = 0

    accuracy_loader = DataLoader(
        modular_train_dataset,
        batch_size=1024,
        shuffle=False
    )

    for (
        grid,
        vector,
        previous_action,
        mask,
        y
    ) in accuracy_loader:

        grid = grid.to(DEVICE)
        vector = vector.to(DEVICE)
        previous_action = previous_action.to(DEVICE)
        mask = mask.to(DEVICE)
        y = y.to(DEVICE)

        logits = modular_model(
            grid,
            vector,
            previous_action
        )

        logits = logits.masked_fill(
            ~mask,
            -1e9
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += int(
            (predictions == y).sum().item()
        )

        total += y.size(0)

    return correct / total

In [13]:
MODULAR_EPOCHS = 30

for epoch in tqdm(
    range(1, MODULAR_EPOCHS + 1)
):
    modular_model.train()

    total_loss = 0.0
    total_examples = 0

    for (
        grid,
        vector,
        previous_action,
        mask,
        y
    ) in modular_train_loader:

        grid = grid.to(DEVICE)
        vector = vector.to(DEVICE)
        previous_action = previous_action.to(DEVICE)
        mask = mask.to(DEVICE)
        y = y.to(DEVICE)

        logits = modular_model(
            grid,
            vector,
            previous_action
        )

        logits = logits.masked_fill(
            ~mask,
            -1e9
        )

        loss = modular_criterion(
            logits,
            y
        )

        modular_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            modular_model.parameters(),
            max_norm=5.0
        )

        modular_optimizer.step()

        batch_size = y.size(0)

        total_loss += (
            float(loss.item())
            * batch_size
        )

        total_examples += batch_size

    average_loss = (
        total_loss
        / total_examples
    )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == MODULAR_EPOCHS
    ):
        train_accuracy = modular_train_accuracy()

        print(
            f"epoch {epoch:02d} | "
            f"loss {average_loss:.4f} | "
            f"train action acc {train_accuracy:.3f}"
        )

  0%|          | 0/30 [00:00<?, ?it/s]

epoch 01 | loss 0.4120 | train action acc 0.940
epoch 05 | loss 0.0184 | train action acc 0.995
epoch 10 | loss 0.0236 | train action acc 0.997
epoch 15 | loss 0.0030 | train action acc 1.000
epoch 20 | loss 0.0217 | train action acc 0.998
epoch 25 | loss 0.0004 | train action acc 1.000
epoch 30 | loss 0.0001 | train action acc 1.000


In [14]:
@torch.no_grad()
def modular_model_action(
    obs,
    previous_action
):
    modular_model.eval()

    grid = torch.tensor(
        add_target_channel(obs),
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    vector = torch.tensor(
        obs["vector"],
        dtype=torch.float32,
        device=DEVICE
    ).unsqueeze(0)

    previous_action_tensor = torch.tensor(
        [previous_action],
        dtype=torch.long,
        device=DEVICE
    )

    mask = torch.tensor(
        obs["action_mask"],
        dtype=torch.bool,
        device=DEVICE
    )

    logits = modular_model(
        grid,
        vector,
        previous_action_tensor
    ).squeeze(0)

    logits = logits.masked_fill(
        ~mask,
        -1e9
    )

    return int(
        logits.argmax().item()
    )
def run_episode_modular(
    scenario,
    max_steps=MAX_STEPS,
    render=False
):
    simulator = DeliverySimulator8x8()
    simulator.reset(scenario)

    previous_action = START_ACTION

    frames = []
    actions = []

    invalid_pickup_or_dropoff = 0
    done = False

    if render:
        frames.append(
            simulator.render()
        )

    for _ in range(max_steps):
        obs = simulator.observation()

        action = modular_model_action(
            obs,
            previous_action
        )

        _, done, timed_out, info = (
            simulator.step(action)
        )

        actions.append(action)

        invalid_pickup_or_dropoff += int(
            info["invalid_pickup_or_dropoff"]
        )

        previous_action = action

        if render:
            frames.append(
                simulator.render()
            )

        if done or timed_out:
            break

    return {
        "success": done,
        "steps": len(actions),
        "actions": actions,
        "frames": frames,
        "invalid_pickup_or_dropoff": (
            invalid_pickup_or_dropoff
        ),
    }
def evaluate_modular_model(scenarios):
    results = [
        run_episode_modular(scenario)
        for scenario in tqdm(
            scenarios,
            desc="Evaluating modular navigation model"
        )
    ]

    return {
        "success_rate": float(
            np.mean([
                result["success"]
                for result in results
            ])
        ),
        "avg_steps": float(
            np.mean([
                result["steps"]
                for result in results
            ])
        ),
        "avg_invalid_pickup_or_dropoff": float(
            np.mean([
                result["invalid_pickup_or_dropoff"]
                for result in results
            ])
        ),
        "results": results,
    }

In [15]:
modular_eval = evaluate_modular_model(valid_scenarios)

print("Modular model:", modular_eval["success_rate"])
print("Average steps:", modular_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    modular_eval["avg_invalid_pickup_or_dropoff"]
)


Evaluating modular navigation model:   0%|          | 0/200 [00:00<?, ?it/s]

Modular model: 0.875
Average steps: 26.545
Invalid pickup/dropoff: 0.0


In [18]:
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)

GRU_USE_ROTATIONS = True

GRU_DELTA_TO_ACTION = {
    delta: action
    for action, delta in ACTION_DELTAS.items()
}


def gru_rotate_coordinate(row, col, rotations):
    """Rotate one coordinate 90 degrees counterclockwise `rotations` times."""
    row = int(row)
    col = int(col)

    for _ in range(rotations % 4):
        row, col = GRID_SIZE - 1 - col, row

    return row, col


def gru_rotate_delta(dr, dc, rotations):
    """Rotate one movement delta 90 degrees counterclockwise."""
    dr = int(dr)
    dc = int(dc)

    for _ in range(rotations % 4):
        dr, dc = -dc, dr

    return dr, dc


def gru_rotate_action(action, rotations):
    """
    Rotate a movement action.

    pickup, dropoff and START_ACTION do not change.
    """
    action = int(action)

    if action not in ACTION_DELTAS:
        return action

    dr, dc = ACTION_DELTAS[action]
    rotated_delta = gru_rotate_delta(
        dr,
        dc,
        rotations,
    )

    return GRU_DELTA_TO_ACTION[rotated_delta]


def gru_rotate_observation(obs, rotations):
    """
    Rotate every spatial component of an observation consistently.

    The same transformation is applied to:
    - grid channels;
    - agent coordinates;
    - current-target coordinates;
    - relative target direction;
    - blocked-direction features;
    - action mask.
    """
    rotations %= 4

    rotated_grid = np.rot90(
        obs["grid"],
        k=rotations,
        axes=(1, 2),
    ).copy()

    old_vector = np.asarray(
        obs["vector"],
        dtype=np.float32,
    )
    new_vector = old_vector.copy()

    row, col, package_field, destination = obs["state"]

    new_row, new_col = gru_rotate_coordinate(
        row,
        col,
        rotations,
    )

    target_row = int(round(
        float(old_vector[5]) * (GRID_SIZE - 1)
    ))
    target_col = int(round(
        float(old_vector[6]) * (GRID_SIZE - 1)
    ))

    new_target_row, new_target_col = gru_rotate_coordinate(
        target_row,
        target_col,
        rotations,
    )

    new_vector[0] = new_row / (GRID_SIZE - 1)
    new_vector[1] = new_col / (GRID_SIZE - 1)

    new_vector[2] = old_vector[2]
    new_vector[3] = old_vector[3]
    new_vector[4] = old_vector[4]

    new_vector[5] = new_target_row / (GRID_SIZE - 1)
    new_vector[6] = new_target_col / (GRID_SIZE - 1)
    new_vector[7] = (
        new_target_row - new_row
    ) / (GRID_SIZE - 1)
    new_vector[8] = (
        new_target_col - new_col
    ) / (GRID_SIZE - 1)

    rotated_blocked = np.zeros(
        4,
        dtype=np.float32,
    )

    for old_action in range(4):
        new_action = gru_rotate_action(
            old_action,
            rotations,
        )
        rotated_blocked[new_action] = old_vector[9 + old_action]

    new_vector[9:13] = rotated_blocked

    old_mask = np.asarray(
        obs["action_mask"],
        dtype=bool,
    )
    new_mask = np.zeros_like(old_mask)

    for old_action in range(6):
        new_action = gru_rotate_action(
            old_action,
            rotations,
        )
        new_mask[new_action] = old_mask[old_action]

    return {
        "grid": rotated_grid,
        "vector": new_vector.astype(np.float32),
        "action_mask": new_mask,
        "state": (
            new_row,
            new_col,
            int(package_field),
            int(destination),
        ),
    }


In [19]:
class SequentialNavigationDataset(Dataset):
    """
    Returns complete expert trajectories instead of independent steps.

    One rotation, when enabled, is applied consistently to every step
    of the same trajectory.
    """

    def __init__(
        self,
        trajectories,
        use_rotations=False,
    ):
        self.items = []
        self.action_counts = Counter()

        rotations = (
            [0, 1, 2, 3]
            if use_rotations
            else [0]
        )

        for trajectory in trajectories:
            for rotation in rotations:
                self.items.append(
                    (trajectory, rotation)
                )

                for action in trajectory["actions"]:
                    rotated_action = gru_rotate_action(
                        int(action),
                        rotation,
                    )
                    self.action_counts[rotated_action] += 1

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        trajectory, rotation = self.items[index]

        grids = []
        vectors = []
        previous_actions = []
        masks = []
        actions = []
        remaining_budgets = []

        previous_action = START_ACTION

        for step_index, (obs, action) in enumerate(zip(
            trajectory["observations"],
            trajectory["actions"],
            strict=True,
        )):
            rotated_obs = gru_rotate_observation(
                obs,
                rotation,
            )
            rotated_action = gru_rotate_action(
                int(action),
                rotation,
            )
            rotated_previous_action = gru_rotate_action(
                previous_action,
                rotation,
            )

            grids.append(torch.tensor(
                add_target_channel(rotated_obs),
                dtype=torch.float32,
            ))
            vectors.append(torch.tensor(
                rotated_obs["vector"],
                dtype=torch.float32,
            ))
            previous_actions.append(
                rotated_previous_action
            )
            masks.append(torch.tensor(
                rotated_obs["action_mask"],
                dtype=torch.bool,
            ))
            actions.append(rotated_action)

            remaining_budget = max(
                0.0,
                (MAX_STEPS - step_index) / MAX_STEPS,
            )
            remaining_budgets.append([
                remaining_budget
            ])

            previous_action = int(action)

        return {
            "grids": torch.stack(grids),
            "vectors": torch.stack(vectors),
            "previous_actions": torch.tensor(
                previous_actions,
                dtype=torch.long,
            ),
            "masks": torch.stack(masks),
            "actions": torch.tensor(
                actions,
                dtype=torch.long,
            ),
            "remaining_budgets": torch.tensor(
                remaining_budgets,
                dtype=torch.float32,
            ),
        }


def collate_navigation_sequences(batch):
    """Pad variable-length trajectories while preserving their lengths."""
    lengths = torch.tensor(
        [item["actions"].size(0) for item in batch],
        dtype=torch.long,
    )

    return {
        "grids": pad_sequence(
            [item["grids"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "vectors": pad_sequence(
            [item["vectors"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "previous_actions": pad_sequence(
            [item["previous_actions"] for item in batch],
            batch_first=True,
            padding_value=START_ACTION,
        ),
        "masks": pad_sequence(
            [item["masks"] for item in batch],
            batch_first=True,
            padding_value=False,
        ),
        "actions": pad_sequence(
            [item["actions"] for item in batch],
            batch_first=True,
            padding_value=-100,
        ),
        "remaining_budgets": pad_sequence(
            [item["remaining_budgets"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "lengths": lengths,
    }


In [20]:
gru_train_dataset = SequentialNavigationDataset(
    train_trajectories,
    use_rotations=GRU_USE_ROTATIONS,
)

gru_train_loader = DataLoader(
    gru_train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_navigation_sequences,
)

gru_example = gru_train_dataset[0]

print("Sequential trajectories:", len(gru_train_dataset))
print("Example length:", gru_example["actions"].size(0))
print("Grid sequence shape:", gru_example["grids"].shape)
print("Vector sequence shape:", gru_example["vectors"].shape)
print("Remaining-budget shape:", gru_example["remaining_budgets"].shape)
print("Rotations enabled:", GRU_USE_ROTATIONS)
print("Action counts:", gru_train_dataset.action_counts)


Sequential trajectories: 1600
Example length: 23
Grid sequence shape: torch.Size([23, 7, 8, 8])
Vector sequence shape: torch.Size([23, 13])
Remaining-budget shape: torch.Size([23, 1])
Rotations enabled: True
Action counts: Counter({1: 4527, 2: 4527, 0: 4527, 3: 4527, 4: 1600, 5: 1600})


In [21]:
class CNNGRUNavigationModel(nn.Module):
    """
    CNN spatial encoder followed by one GRU and one action head.

    The GRU hidden state acts as learned temporal memory. The model receives
    complete trajectories during training and one step at a time during
    inference.
    """

    def __init__(
        self,
        vector_dim=13,
        num_actions=6,
        previous_action_vocab=7,
        hidden_dim=192,
    ):
        super().__init__()

        self.spatial_encoder = nn.Sequential(
            nn.Conv2d(
                7,
                32,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 32),
            nn.ReLU(),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv2d(
                64,
                96,
                kernel_size=3,
                padding=1,
            ),
            nn.GroupNorm(8, 96),
            nn.ReLU(),
        )

        self.vector_encoder = nn.Sequential(
            nn.Linear(vector_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.15),

            nn.Linear(64, 64),
            nn.ReLU(),
        )

        self.previous_action_embedding = nn.Embedding(
            previous_action_vocab,
            16,
        )
        self.full_map_encoder = nn.Sequential(
            nn.Conv2d(
                96,
                16,
                kernel_size=1,
            ),
            nn.ReLU(),
            nn.Flatten(),
        )
        local_spatial_dim = 96 * 7
        full_map_dim = 16 * GRID_SIZE * GRID_SIZE
        
        spatial_decision_dim = (
            local_spatial_dim
            + full_map_dim
        )

        combined_dim = (
            spatial_decision_dim
            + 64
            + 16
            + 1
        )

        self.input_projection = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256, hidden_dim),
            nn.ReLU(),
        )

        self.gru = nn.GRU(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
        )

        self.output_dropout = nn.Dropout(0.15)
        self.action_head = nn.Linear(
            hidden_dim,
            num_actions,
        )
    @staticmethod
    def gather_position_features(
        feature_map,
        rows,
        columns,
    ):
        batch_indices = torch.arange(
            feature_map.size(0),
            device=feature_map.device,
        )

        return feature_map[
            batch_indices,
            :,
            rows,
            columns,
        ]

    def encode_steps(
        self,
        grids,
        vectors,
        previous_actions,
        remaining_budgets,
    ):
        feature_map = self.spatial_encoder(
            grids
        )

        height = feature_map.size(2)
        width = feature_map.size(3)

        robot_rows = torch.round(
            vectors[:, 0] * (height - 1)
        ).long().clamp(
            min=0,
            max=height - 1,
        )

        robot_columns = torch.round(
            vectors[:, 1] * (width - 1)
        ).long().clamp(
            min=0,
            max=width - 1,
        )

        target_rows = torch.round(
            vectors[:, 5] * (height - 1)
        ).long().clamp(
            min=0,
            max=height - 1,
        )

        target_columns = torch.round(
            vectors[:, 6] * (width - 1)
        ).long().clamp(
            min=0,
            max=width - 1,
        )

        robot_features = self.gather_position_features(
            feature_map,
            robot_rows,
            robot_columns,
        )

        target_features = self.gather_position_features(
            feature_map,
            target_rows,
            target_columns,
        )

        neighbor_features = []

        for action_id in range(4):
            dr, dc = ACTION_DELTAS[action_id]

            neighbor_rows = robot_rows + dr
            neighbor_columns = robot_columns + dc

            valid_position = (
                (neighbor_rows >= 0)
                & (neighbor_rows < height)
                & (neighbor_columns >= 0)
                & (neighbor_columns < width)
            )

            safe_rows = neighbor_rows.clamp(
                min=0,
                max=height - 1,
            )
            safe_columns = neighbor_columns.clamp(
                min=0,
                max=width - 1,
            )

            current_features = self.gather_position_features(
                feature_map,
                safe_rows,
                safe_columns,
            )

            current_features = (
                current_features
                * valid_position.unsqueeze(1).to(
                    feature_map.dtype
                )
            )

            neighbor_features.append(
                current_features
            )

        neighbor_features = torch.cat(
            neighbor_features,
            dim=1,
        )

        global_features = feature_map.mean(
            dim=(2, 3)
        )
        full_map_features = self.full_map_encoder(
            feature_map
        )
        vector_features = self.vector_encoder(
            vectors
        )

        previous_action_features = (
            self.previous_action_embedding(
                previous_actions
            )
        )

        combined = torch.cat(
            [
                robot_features,
                target_features,
                neighbor_features,
                global_features,
                full_map_features,
                vector_features,
                previous_action_features,
                remaining_budgets,
            ],
            dim=1,
        )
        

        return self.input_projection(
            combined
        )

    def forward(
        self,
        grids,
        vectors,
        previous_actions,
        remaining_budgets,
        lengths=None,
        hidden=None,
    ):
        batch_size, sequence_length = grids.shape[:2]

        flat_step_features = self.encode_steps(
            grids.reshape(
                batch_size * sequence_length,
                *grids.shape[2:],
            ),
            vectors.reshape(
                batch_size * sequence_length,
                vectors.size(-1),
            ),
            previous_actions.reshape(-1),
            remaining_budgets.reshape(-1, 1),
        )

        sequence_features = flat_step_features.reshape(
            batch_size,
            sequence_length,
            -1,
        )

        if lengths is not None:
            packed = pack_padded_sequence(
                sequence_features,
                lengths.cpu(),
                batch_first=True,
                enforce_sorted=False,
            )

            packed_output, new_hidden = self.gru(
                packed,
                hidden,
            )

            recurrent_output, _ = pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=sequence_length,
            )
        else:
            recurrent_output, new_hidden = self.gru(
                sequence_features,
                hidden,
            )

        logits = self.action_head(
            self.output_dropout(
                recurrent_output
            )
        )

        return logits, new_hidden


In [22]:
gru_model = CNNGRUNavigationModel(
    vector_dim=gru_example["vectors"].size(-1),
    num_actions=6,
    hidden_dim=192,
).to(DEVICE)

gru_counts = torch.zeros(
    6,
    dtype=torch.float32,
)

for action_id, count in gru_train_dataset.action_counts.items():
    gru_counts[action_id] = count

gru_class_weights = (
    gru_counts.sum()
    / (
        6
        * gru_counts.clamp(min=1)
    )
)
gru_class_weights = (
    gru_class_weights
    / gru_class_weights.mean()
).to(DEVICE)

gru_optimizer = torch.optim.AdamW(
    gru_model.parameters(),
    lr=5e-4,
    weight_decay=1e-4,
)

gru_criterion = nn.CrossEntropyLoss(
    weight=gru_class_weights,
    ignore_index=-100,
)

print(gru_model)
print(
    "GRU parameters:",
    sum(
        parameter.numel()
        for parameter in gru_model.parameters()
    ),
)
print("GRU class weights:", gru_class_weights)


CNNGRUNavigationModel(
  (spatial_encoder): Sequential(
    (0): Conv2d(7, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): ReLU()
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): GroupNorm(8, 64, eps=1e-05, affine=True)
    (5): ReLU()
    (6): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): GroupNorm(8, 64, eps=1e-05, affine=True)
    (8): ReLU()
    (9): Conv2d(64, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): GroupNorm(8, 96, eps=1e-05, affine=True)
    (11): ReLU()
  )
  (vector_encoder): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): ReLU()
  )
  (previous_action_embedding): Embedding(7, 16)
  (full_map_encoder): Sequential(
    (0): Conv2d(96, 16, kernel_size=(1, 1), stride=(1, 1))
    (1

In [23]:
@torch.no_grad()
def gru_train_accuracy():
    gru_model.eval()

    correct = 0
    total = 0

    accuracy_loader = DataLoader(
        gru_train_dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=collate_navigation_sequences,
    )

    for batch in accuracy_loader:
        grids = batch["grids"].to(DEVICE)
        vectors = batch["vectors"].to(DEVICE)
        previous_actions = batch["previous_actions"].to(DEVICE)
        masks = batch["masks"].to(DEVICE)
        actions = batch["actions"].to(DEVICE)
        remaining_budgets = batch["remaining_budgets"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)

        logits, _ = gru_model(
            grids,
            vectors,
            previous_actions,
            remaining_budgets,
            lengths=lengths,
        )

        masked_logits = logits.masked_fill(
            ~masks,
            -1e9,
        )

        predictions = masked_logits.argmax(
            dim=-1
        )

        valid_steps = actions != -100

        correct += int(
            (
                (predictions == actions)
                & valid_steps
            ).sum().item()
        )
        total += int(
            valid_steps.sum().item()
        )

    return correct / max(total, 1)


In [24]:
GRU_EPOCHS = 25

for epoch in tqdm(
    range(1, GRU_EPOCHS + 1),
    desc="Training CNN + GRU",
):
    gru_model.train()

    total_loss = 0.0
    total_valid_steps = 0

    for batch in gru_train_loader:
        grids = batch["grids"].to(DEVICE)
        vectors = batch["vectors"].to(DEVICE)
        previous_actions = batch["previous_actions"].to(DEVICE)
        masks = batch["masks"].to(DEVICE)
        actions = batch["actions"].to(DEVICE)
        remaining_budgets = batch["remaining_budgets"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)

        logits, _ = gru_model(
            grids,
            vectors,
            previous_actions,
            remaining_budgets,
            lengths=lengths,
        )

        masked_logits = logits.masked_fill(
            ~masks,
            -1e9,
        )

        loss = gru_criterion(
            masked_logits.reshape(-1, 6),
            actions.reshape(-1),
        )

        gru_optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            gru_model.parameters(),
            max_norm=2.0,
        )

        gru_optimizer.step()

        valid_steps = int(
            (actions != -100).sum().item()
        )

        total_loss += (
            float(loss.item())
            * valid_steps
        )
        total_valid_steps += valid_steps

    average_loss = (
        total_loss
        / max(total_valid_steps, 1)
    )

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == GRU_EPOCHS
    ):
        train_accuracy = gru_train_accuracy()

        print(
            f"epoch {epoch:02d} | "
            f"loss {average_loss:.4f} | "
            f"train action acc {train_accuracy:.3f}"
        )


Training CNN + GRU:   0%|          | 0/25 [00:00<?, ?it/s]

epoch 01 | loss 0.5082 | train action acc 0.866
epoch 05 | loss 0.1569 | train action acc 0.928
epoch 10 | loss 0.0651 | train action acc 0.982
epoch 15 | loss 0.0276 | train action acc 0.995
epoch 20 | loss 0.0081 | train action acc 0.999
epoch 25 | loss 0.0137 | train action acc 0.998


In [25]:
@torch.no_grad()
def gru_model_action(
    obs,
    previous_action,
    remaining_budget,
    hidden,
):
    """
    Predict one action and return the updated GRU memory.

    Only the environment action mask is applied.
    """
    gru_model.eval()

    grid = torch.tensor(
        add_target_channel(obs),
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    vector = torch.tensor(
        obs["vector"],
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    previous_action_tensor = torch.tensor(
        [[previous_action]],
        dtype=torch.long,
        device=DEVICE,
    )

    remaining_budget_tensor = torch.tensor(
        [[[remaining_budget]]],
        dtype=torch.float32,
        device=DEVICE,
    )

    logits, new_hidden = gru_model(
        grid,
        vector,
        previous_action_tensor,
        remaining_budget_tensor,
        hidden=hidden,
    )

    logits = logits[0, 0]

    mask = torch.tensor(
        obs["action_mask"],
        dtype=torch.bool,
        device=DEVICE,
    )

    logits = logits.masked_fill(
        ~mask,
        -1e9,
    )

    action = int(
        logits.argmax().item()
    )

    return action, new_hidden


def run_episode_gru(
    scenario,
    max_steps=MAX_STEPS,
    render=False,
):
    simulator = DeliverySimulator8x8()
    simulator.reset(scenario)

    previous_action = START_ACTION
    hidden = None

    frames = []
    actions = []

    invalid_pickup_or_dropoff = 0
    done = False

    if render:
        frames.append(
            simulator.render()
        )

    for step_index in range(max_steps):
        obs = simulator.observation()

        remaining_budget = max(
            0.0,
            (max_steps - step_index) / max_steps,
        )

        action, hidden = gru_model_action(
            obs,
            previous_action,
            remaining_budget,
            hidden,
        )

        _, done, timed_out, info = (
            simulator.step(action)
        )

        actions.append(action)

        invalid_pickup_or_dropoff += int(
            info["invalid_pickup_or_dropoff"]
        )

        previous_action = action

        if render:
            frames.append(
                simulator.render()
            )

        if done or timed_out:
            break

    return {
        "success": done,
        "steps": len(actions),
        "actions": actions,
        "frames": frames,
        "invalid_pickup_or_dropoff": (
            invalid_pickup_or_dropoff
        ),
    }


def evaluate_gru_model(scenarios):
    results = [
        run_episode_gru(scenario)
        for scenario in tqdm(
            scenarios,
            desc="Evaluating CNN + GRU",
        )
    ]

    return {
        "success_rate": float(
            np.mean([
                result["success"]
                for result in results
            ])
        ),
        "avg_steps": float(
            np.mean([
                result["steps"]
                for result in results
            ])
        ),
        "avg_invalid_pickup_or_dropoff": float(
            np.mean([
                result["invalid_pickup_or_dropoff"]
                for result in results
            ])
        ),
        "results": results,
    }


In [26]:
gru_eval = evaluate_gru_model(
    valid_scenarios
)

print("CNN + GRU:", gru_eval["success_rate"])
print("Average steps:", gru_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    gru_eval["avg_invalid_pickup_or_dropoff"],
)

print()
print("Baseline modular:", modular_eval["success_rate"])
print("CNN + GRU:", gru_eval["success_rate"])


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

CNN + GRU: 0.945
Average steps: 19.235
Invalid pickup/dropoff: 0.0

Baseline modular: 0.875
CNN + GRU: 0.945


In [27]:
successful_episodes = sum(
    result["success"]
    for result in gru_eval["results"]
)

total_episodes = len(gru_eval["results"])

score = successful_episodes / total_episodes

print(f"Successful episodes: {successful_episodes}/{total_episodes}")
print(f"Score: {score:.4f}")
print(f"Score percentage: {score * 100:.2f}%")

Successful episodes: 189/200
Score: 0.9450
Score percentage: 94.50%


In [28]:
modular_eval = evaluate_modular_model(valid_scenarios)

print("Score:", modular_eval["success_rate"])
print("Average steps:", modular_eval["avg_steps"])
print(
    "Invalid pickup/dropoff:",
    modular_eval["avg_invalid_pickup_or_dropoff"]
)

Evaluating modular navigation model:   0%|          | 0/200 [00:00<?, ?it/s]

Score: 0.875
Average steps: 26.545
Invalid pickup/dropoff: 0.0


In [29]:
successful_episodes = sum(
    result["success"]
    for result in modular_eval["results"]
)

total_episodes = len(modular_eval["results"])

score = successful_episodes / total_episodes

print(f"Successful episodes: {successful_episodes}/{total_episodes}")
print(f"Score: {score:.4f}")
print(f"Score percentage: {score * 100:.2f}%")

Successful episodes: 175/200
Score: 0.8750
Score percentage: 87.50%


In [30]:
# =============================================================================
# 1. Proteger el baseline verdadero antes de DAgger
# =============================================================================

import copy
import pickle
from collections import deque

import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler

BASELINE_SUCCESS = float(gru_eval["success_rate"])
BASELINE_AVG_STEPS = float(gru_eval["avg_steps"])

baseline_checkpoint_path = (
    OUTPUT_DIR
    / f"cnn_gru_true_baseline_{BASELINE_SUCCESS:.3f}.pt"
)

baseline_checkpoint = {
    "model_state_dict": copy.deepcopy(
        gru_model.state_dict()
    ),
    "success_rate": BASELINE_SUCCESS,
    "avg_steps": BASELINE_AVG_STEPS,
    "stage": "baseline",
    "round": 0,
    "epoch": 0,
}

torch.save(
    baseline_checkpoint,
    baseline_checkpoint_path,
)

best_recovery_checkpoint_path = (
    OUTPUT_DIR
    / "cnn_gru_perturbed_dagger_best.pt"
)

torch.save(
    baseline_checkpoint,
    best_recovery_checkpoint_path,
)

print("Protected baseline:", baseline_checkpoint_path)
print("Baseline success:", BASELINE_SUCCESS)
print("Baseline average steps:", BASELINE_AVG_STEPS)


Protected baseline: /kaggle/working/cnn_gru_true_baseline_0.945.pt
Baseline success: 0.945
Baseline average steps: 19.235


In [31]:
# =============================================================================
# 2. Oráculo BFS usado solo durante la recolección de entrenamiento
# =============================================================================


def shortest_distance_map(simulator, target_position):
    """Distancia mínima desde cada celda transitable hasta target_position."""
    distances = np.full(
        (GRID_SIZE, GRID_SIZE),
        fill_value=np.inf,
        dtype=np.float32,
    )

    target_row, target_col = target_position

    if not simulator.can_enter(target_row, target_col):
        return distances

    queue = deque([(target_row, target_col)])
    distances[target_row, target_col] = 0.0

    while queue:
        row, col = queue.popleft()
        next_distance = distances[row, col] + 1.0

        for dr, dc in ACTION_DELTAS.values():
            next_row = row + dr
            next_col = col + dc

            if not simulator.can_enter(next_row, next_col):
                continue

            if distances[next_row, next_col] <= next_distance:
                continue

            distances[next_row, next_col] = next_distance
            queue.append((next_row, next_col))

    return distances


def optimal_actions_for_current_state(simulator):
    """Todas las acciones que reducen en uno la distancia BFS al objetivo."""
    row, col, package_field, destination = simulator.state()
    carrying = package_field == N_DEPOTS

    if (
        not carrying
        and simulator.agent_pos
        == simulator.depots[simulator.package_location]
    ):
        return [4]

    if (
        carrying
        and simulator.agent_pos
        == simulator.depots[destination]
    ):
        return [5]

    target_position = (
        simulator.depots[destination]
        if carrying
        else simulator.depots[simulator.package_location]
    )

    distances = shortest_distance_map(
        simulator,
        target_position,
    )

    current_distance = distances[row, col]
    optimal_actions = []

    for action, (dr, dc) in ACTION_DELTAS.items():
        next_row = row + dr
        next_col = col + dc

        if not simulator.can_enter(next_row, next_col):
            continue

        if distances[next_row, next_col] == current_distance - 1.0:
            optimal_actions.append(action)

    if optimal_actions:
        return optimal_actions

    # Respaldo defensivo; normalmente nunca se usa.
    valid_actions = np.flatnonzero(
        simulator.valid_action_mask()
    ).tolist()

    if not valid_actions:
        raise RuntimeError("El estado no tiene acciones válidas")

    return valid_actions


def controlled_nonoptimal_action(
    simulator,
    optimal_actions,
    rng,
):
    """
    Escoge una acción de movimiento válida, pero no óptima.

    Entre las alternativas se prefieren las que dejan al robot menos lejos del
    objetivo. Así la perturbación es útil, pero no destruye el episodio.
    """
    valid_mask = simulator.valid_action_mask()

    candidates = [
        action
        for action in range(4)
        if valid_mask[action]
        and action not in optimal_actions
    ]

    if not candidates:
        return None

    row, col, package_field, destination = simulator.state()
    carrying = package_field == N_DEPOTS

    target_position = (
        simulator.depots[destination]
        if carrying
        else simulator.depots[simulator.package_location]
    )

    distances = shortest_distance_map(
        simulator,
        target_position,
    )

    candidate_distances = {}

    for action in candidates:
        dr, dc = ACTION_DELTAS[action]
        candidate_distances[action] = float(
            distances[row + dr, col + dc]
        )

    best_nonoptimal_distance = min(
        candidate_distances.values()
    )

    safest_candidates = [
        action
        for action in candidates
        if candidate_distances[action]
        == best_nonoptimal_distance
    ]

    return int(rng.choice(safest_candidates))


In [32]:
# =============================================================================
# 3. Recolección de trayectorias con una perturbación forzada
# =============================================================================


@torch.no_grad()
def gru_policy_logits(
    obs,
    previous_action,
    remaining_budget,
    hidden,
):
    """Logits enmascarados y nuevo hidden para un solo paso."""
    gru_model.eval()

    grid = torch.tensor(
        add_target_channel(obs),
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    vector = torch.tensor(
        obs["vector"],
        dtype=torch.float32,
        device=DEVICE,
    ).unsqueeze(0).unsqueeze(0)

    previous_action_tensor = torch.tensor(
        [[previous_action]],
        dtype=torch.long,
        device=DEVICE,
    )

    remaining_budget_tensor = torch.tensor(
        [[[remaining_budget]]],
        dtype=torch.float32,
        device=DEVICE,
    )

    logits, new_hidden = gru_model(
        grid,
        vector,
        previous_action_tensor,
        remaining_budget_tensor,
        hidden=hidden,
    )

    logits = logits[0, 0]

    valid_mask = torch.tensor(
        obs["action_mask"],
        dtype=torch.bool,
        device=DEVICE,
    )

    logits = logits.masked_fill(
        ~valid_mask,
        -1e9,
    )

    return logits, new_hidden


@torch.no_grad()
def collect_perturbed_recovery_trajectory(
    scenario,
    seed,
    recovery_horizon=12,
    min_perturbation_step=1,
    max_perturbation_step=6,
    max_steps=MAX_STEPS,
):
    """
    Ejecuta la política y fuerza exactamente una desviación cuando es posible.

    - observations: estados realmente visitados;
    - actions: etiqueta BFS para cada estado;
    - executed_actions: acciones que realmente produjeron la trayectoria;
    - loss_weights: concentra el entrenamiento después de la perturbación.
    """
    rng = random.Random(seed)

    simulator = DeliverySimulator8x8()
    simulator.reset(scenario)

    previous_action = START_ACTION
    hidden = None

    observations = []
    oracle_actions = []
    executed_actions = []
    loss_weights = []
    disagreement_mask = []

    planned_step = rng.randint(
        min_perturbation_step,
        max_perturbation_step,
    )

    perturbation_index = None
    recovery_steps = 0
    done = False

    for step_index in range(max_steps):
        obs = simulator.observation()

        remaining_budget = max(
            0.0,
            (max_steps - step_index) / max_steps,
        )

        logits, new_hidden = gru_policy_logits(
            obs=obs,
            previous_action=previous_action,
            remaining_budget=remaining_budget,
            hidden=hidden,
        )

        model_action = int(logits.argmax().item())

        optimal_actions = optimal_actions_for_current_state(
            simulator
        )

        # Conserva un único desempate consistente: entre rutas mínimas se usa
        # la que la política ya prefiere. Las acciones óptimas múltiples ya se
        # probaron por separado y no mejoraron el score.
        oracle_action = int(max(
            optimal_actions,
            key=lambda action: float(logits[action].item()),
        ))

        just_perturbed = False

        if (
            perturbation_index is None
            and step_index >= planned_step
        ):
            perturbation_action = controlled_nonoptimal_action(
                simulator=simulator,
                optimal_actions=optimal_actions,
                rng=rng,
            )

            if perturbation_action is not None:
                executed_action = perturbation_action
                perturbation_index = len(observations)
                just_perturbed = True
            else:
                executed_action = model_action
        else:
            executed_action = model_action

        model_disagrees = model_action not in optimal_actions

        if perturbation_index is None:
            # El prefijo solo existe para reconstruir correctamente el hidden.
            step_weight = 0.0
        elif just_perturbed:
            # La política no eligió este error; fue forzado deliberadamente.
            step_weight = 0.25
        else:
            # Estos son los estados de recuperación que sí queremos aprender.
            step_weight = 8.0 if model_disagrees else 3.0
            recovery_steps += 1

        observations.append(obs)
        oracle_actions.append(oracle_action)
        executed_actions.append(int(executed_action))
        loss_weights.append(float(step_weight))
        disagreement_mask.append(bool(model_disagrees))

        _, done, timed_out, _ = simulator.step(
            executed_action
        )

        previous_action = int(executed_action)
        hidden = new_hidden

        if done or timed_out:
            break

        if (
            perturbation_index is not None
            and recovery_steps >= recovery_horizon
        ):
            break

    if perturbation_index is None:
        return None

    return {
        "scenario": scenario,
        "observations": observations,
        "actions": oracle_actions,
        "executed_actions": executed_actions,
        "loss_weights": loss_weights,
        "disagreement_mask": disagreement_mask,
        "perturbation_index": perturbation_index,
        "num_steps": len(observations),
        "success": done,
        "post_perturbation_disagreements": int(sum(
            disagreement_mask[perturbation_index + 1:]
        )),
    }


@torch.no_grad()
def collect_perturbed_dagger_round(
    trajectories,
    round_index,
    recovery_horizon=12,
):
    """Una trayectoria perturbada por cada escenario de entrenamiento."""
    collected = []

    for trajectory_index, trajectory in enumerate(tqdm(
        trajectories,
        desc=f"Perturbed DAgger collection round {round_index}",
    )):
        recovery_trajectory = (
            collect_perturbed_recovery_trajectory(
                scenario=trajectory["scenario"],
                seed=(
                    SEED
                    + 100_000 * round_index
                    + trajectory_index
                ),
                recovery_horizon=recovery_horizon,
            )
        )

        if recovery_trajectory is not None:
            collected.append(recovery_trajectory)

    statistics = {
        "collected": len(collected),
        "mean_length": float(np.mean([
            trajectory["num_steps"]
            for trajectory in collected
        ])) if collected else 0.0,
        "mean_recovery_disagreements": float(np.mean([
            trajectory["post_perturbation_disagreements"]
            for trajectory in collected
        ])) if collected else 0.0,
        "rollout_success_rate": float(np.mean([
            trajectory["success"]
            for trajectory in collected
        ])) if collected else 0.0,
    }

    return collected, statistics


In [34]:
# =============================================================================
# 4. Dataset que separa etiquetas expertas de acciones ejecutadas
# =============================================================================


class RecoveryFineTuneDataset(Dataset):
    """
    Dataset común para demostraciones originales y trayectorias de recuperación.

    En DAgger, previous_action se construye desde executed_actions, mientras que
    el objetivo de clasificación sale de actions (BFS).
    """

    def __init__(
        self,
        original_trajectories,
        recovery_trajectories,
        use_rotations=True,
    ):
        self.items = []

        rotations = [0, 1, 2, 3] if use_rotations else [0]

        for trajectory in original_trajectories:
            for rotation in rotations:
                self.items.append({
                    "trajectory": trajectory,
                    "rotation": rotation,
                    "source": "original",
                })

        self.original_item_count = len(self.items)

        for trajectory in recovery_trajectories:
            for rotation in rotations:
                self.items.append({
                    "trajectory": trajectory,
                    "rotation": rotation,
                    "source": "recovery",
                })

        self.recovery_item_count = (
            len(self.items) - self.original_item_count
        )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        item = self.items[index]
        trajectory = item["trajectory"]
        rotation = item["rotation"]
        source = item["source"]

        observations = trajectory["observations"]
        target_actions = trajectory["actions"]
        executed_actions = trajectory.get(
            "executed_actions",
            target_actions,
        )
        stored_weights = trajectory.get(
            "loss_weights"
        )

        if not (
            len(observations)
            == len(target_actions)
            == len(executed_actions)
        ):
            raise ValueError(
                "observations, actions y executed_actions "
                "deben tener la misma longitud"
            )

        grids = []
        vectors = []
        previous_actions = []
        masks = []
        actions = []
        remaining_budgets = []
        loss_weights = []

        previous_executed_action = START_ACTION

        for step_index, (
            obs,
            target_action,
            executed_action,
        ) in enumerate(zip(
            observations,
            target_actions,
            executed_actions,
            strict=True,
        )):
            rotated_obs = gru_rotate_observation(
                obs,
                rotation,
            )

            rotated_target_action = gru_rotate_action(
                int(target_action),
                rotation,
            )

            rotated_previous_action = gru_rotate_action(
                previous_executed_action,
                rotation,
            )

            grids.append(torch.tensor(
                add_target_channel(rotated_obs),
                dtype=torch.float32,
            ))

            vectors.append(torch.tensor(
                rotated_obs["vector"],
                dtype=torch.float32,
            ))

            previous_actions.append(
                rotated_previous_action
            )

            masks.append(torch.tensor(
                rotated_obs["action_mask"],
                dtype=torch.bool,
            ))

            actions.append(rotated_target_action)

            remaining_budgets.append([
                max(
                    0.0,
                    (MAX_STEPS - step_index) / MAX_STEPS,
                )
            ])

            if stored_weights is None:
                loss_weights.append(1.0)
            else:
                loss_weights.append(float(
                    stored_weights[step_index]
                ))

            previous_executed_action = int(
                executed_action
            )

        return {
            "grids": torch.stack(grids),
            "vectors": torch.stack(vectors),
            "previous_actions": torch.tensor(
                previous_actions,
                dtype=torch.long,
            ),
            "masks": torch.stack(masks),
            "actions": torch.tensor(
                actions,
                dtype=torch.long,
            ),
            "remaining_budgets": torch.tensor(
                remaining_budgets,
                dtype=torch.float32,
            ),
            "loss_weights": torch.tensor(
                loss_weights,
                dtype=torch.float32,
            ),
            "source": source,
        }


def collate_recovery_sequences(batch):
    lengths = torch.tensor(
        [item["actions"].size(0) for item in batch],
        dtype=torch.long,
    )

    return {
        "grids": pad_sequence(
            [item["grids"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "vectors": pad_sequence(
            [item["vectors"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "previous_actions": pad_sequence(
            [item["previous_actions"] for item in batch],
            batch_first=True,
            padding_value=START_ACTION,
        ),
        "masks": pad_sequence(
            [item["masks"] for item in batch],
            batch_first=True,
            padding_value=False,
        ),
        "actions": pad_sequence(
            [item["actions"] for item in batch],
            batch_first=True,
            padding_value=-100,
        ),
        "remaining_budgets": pad_sequence(
            [item["remaining_budgets"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "loss_weights": pad_sequence(
            [item["loss_weights"] for item in batch],
            batch_first=True,
            padding_value=0.0,
        ),
        "lengths": lengths,
    }


def make_recovery_loader(
    original_trajectories,
    recovery_trajectories,
    original_probability=0.80,
    batch_size=16,
):
    dataset = RecoveryFineTuneDataset(
        original_trajectories=original_trajectories,
        recovery_trajectories=recovery_trajectories,
        use_rotations=True,
    )

    if dataset.recovery_item_count == 0:
        raise ValueError(
            "No se recolectaron trayectorias de recuperación"
        )

    recovery_probability = 1.0 - original_probability

    sample_weights = torch.empty(
        len(dataset),
        dtype=torch.double,
    )

    sample_weights[:dataset.original_item_count] = (
        original_probability
        / dataset.original_item_count
    )

    sample_weights[dataset.original_item_count:] = (
        recovery_probability
        / dataset.recovery_item_count
    )

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(dataset),
        replacement=True,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=collate_recovery_sequences,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    return dataset, loader


In [35]:
# =============================================================================
# 5. Fine-tuning conservador centrado en recuperación
# =============================================================================


def configure_recovery_fine_tuning():
    """Congela percepción; adapta memoria, acción previa y decisión."""
    for parameter in gru_model.parameters():
        parameter.requires_grad = True

    frozen_modules = [
        gru_model.spatial_encoder,
        gru_model.vector_encoder,
        gru_model.full_map_encoder,
    ]

    for module in frozen_modules:
        for parameter in module.parameters():
            parameter.requires_grad = False

    trainable_parameters = [
        parameter
        for parameter in gru_model.parameters()
        if parameter.requires_grad
    ]

    return frozen_modules, trainable_parameters


def weighted_recovery_loss(
    masked_logits,
    actions,
    loss_weights,
):
    """Cross entropy por paso, ponderada alrededor de la perturbación."""
    batch_size, sequence_length, num_actions = (
        masked_logits.shape
    )

    flat_loss = F.cross_entropy(
        masked_logits.reshape(-1, num_actions),
        actions.reshape(-1),
        ignore_index=-100,
        reduction="none",
    )

    per_step_loss = flat_loss.reshape(
        batch_size,
        sequence_length,
    )

    valid_steps = actions != -100

    effective_weights = (
        loss_weights
        * valid_steps.float()
    )

    denominator = effective_weights.sum().clamp(
        min=1.0
    )

    return (
        per_step_loss * effective_weights
    ).sum() / denominator


def train_one_recovery_epoch(
    loader,
    optimizer,
    frozen_modules,
):
    gru_model.train()

    # Los módulos congelados permanecen en eval para que su Dropout no cambie
    # las representaciones que produjeron el baseline.
    for module in frozen_modules:
        module.eval()

    total_weighted_loss = 0.0
    total_weight = 0.0

    for batch in loader:
        grids = batch["grids"].to(
            DEVICE,
            non_blocking=True,
        )
        vectors = batch["vectors"].to(
            DEVICE,
            non_blocking=True,
        )
        previous_actions = batch[
            "previous_actions"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        masks = batch["masks"].to(
            DEVICE,
            non_blocking=True,
        )
        actions = batch["actions"].to(
            DEVICE,
            non_blocking=True,
        )
        remaining_budgets = batch[
            "remaining_budgets"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        loss_weights = batch[
            "loss_weights"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        lengths = batch["lengths"]

        logits, _ = gru_model(
            grids,
            vectors,
            previous_actions,
            remaining_budgets,
            lengths=lengths,
        )

        masked_logits = logits.masked_fill(
            ~masks,
            -1e9,
        )

        loss = weighted_recovery_loss(
            masked_logits=masked_logits,
            actions=actions,
            loss_weights=loss_weights,
        )

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [
                parameter
                for parameter in gru_model.parameters()
                if parameter.requires_grad
            ],
            max_norm=1.0,
        )

        optimizer.step()

        batch_weight = float(
            (
                loss_weights
                * (actions != -100).float()
            ).sum().item()
        )

        total_weighted_loss += (
            float(loss.item()) * batch_weight
        )
        total_weight += batch_weight

    return (
        total_weighted_loss
        / max(total_weight, 1.0)
    )


PERTURBED_DAGGER_ROUNDS = 2
RECOVERY_HORIZON = 12
DAGGER_EPOCHS_PER_ROUND = 3
DAGGER_LEARNING_RATE = 5e-5
ORIGINAL_TRAJECTORY_PROBABILITY = 0.80

best_recovery_success = BASELINE_SUCCESS
best_recovery_avg_steps = BASELINE_AVG_STEPS
all_recovery_trajectories = []
recovery_history = []

for round_index in range(1, PERTURBED_DAGGER_ROUNDS + 1):
    # Cada ronda parte del mejor checkpoint conocido.
    best_checkpoint = torch.load(
        best_recovery_checkpoint_path,
        map_location=DEVICE,
    )

    gru_model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    new_recovery_trajectories, collection_stats = (
        collect_perturbed_dagger_round(
            trajectories=train_trajectories,
            round_index=round_index,
            recovery_horizon=RECOVERY_HORIZON,
        )
    )

    all_recovery_trajectories.extend(
        new_recovery_trajectories
    )

    print(
        f"Round {round_index} collection | "
        f"trajectories {collection_stats['collected']} | "
        f"mean length {collection_stats['mean_length']:.2f} | "
        f"recovery disagreements "
        f"{collection_stats['mean_recovery_disagreements']:.2f}"
    )

    recovery_dataset, recovery_loader = (
        make_recovery_loader(
            original_trajectories=train_trajectories,
            recovery_trajectories=(
                all_recovery_trajectories
            ),
            original_probability=(
                ORIGINAL_TRAJECTORY_PROBABILITY
            ),
            batch_size=16,
        )
    )

    frozen_modules, trainable_parameters = (
        configure_recovery_fine_tuning()
    )

    recovery_optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=DAGGER_LEARNING_RATE,
        weight_decay=1e-4,
    )

    print(
        "Fine-tune items | original:",
        recovery_dataset.original_item_count,
        "recovery:",
        recovery_dataset.recovery_item_count,
    )

    for dagger_epoch in range(
        1,
        DAGGER_EPOCHS_PER_ROUND + 1,
    ):
        recovery_loss = train_one_recovery_epoch(
            loader=recovery_loader,
            optimizer=recovery_optimizer,
            frozen_modules=frozen_modules,
        )

        validation = evaluate_gru_model(
            valid_scenarios
        )

        success_rate = validation["success_rate"]
        average_steps = validation["avg_steps"]

        recovery_history.append({
            "round": round_index,
            "epoch": dagger_epoch,
            "loss": recovery_loss,
            "success_rate": success_rate,
            "avg_steps": average_steps,
            "recovery_trajectories": len(
                all_recovery_trajectories
            ),
        })

        print(
            f"Perturbed DAgger round {round_index} | "
            f"epoch {dagger_epoch:02d} | "
            f"loss {recovery_loss:.5f} | "
            f"valid success {success_rate:.3f} | "
            f"avg steps {average_steps:.2f}"
        )

        improved = (
            success_rate > best_recovery_success
            or (
                success_rate == best_recovery_success
                and average_steps < best_recovery_avg_steps
            )
        )

        if improved:
            best_recovery_success = success_rate
            best_recovery_avg_steps = average_steps

            torch.save(
                {
                    "model_state_dict": copy.deepcopy(
                        gru_model.state_dict()
                    ),
                    "success_rate": success_rate,
                    "avg_steps": average_steps,
                    "stage": "perturbed_dagger",
                    "round": round_index,
                    "epoch": dagger_epoch,
                },
                best_recovery_checkpoint_path,
            )

            print(
                "Saved new best recovery checkpoint:",
                best_recovery_checkpoint_path,
            )

    # No se arrastra una época peor a la siguiente ronda.
    best_checkpoint = torch.load(
        best_recovery_checkpoint_path,
        map_location=DEVICE,
    )

    gru_model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

with (
    OUTPUT_DIR / "perturbed_dagger_trajectories.pkl"
).open("wb") as file:
    pickle.dump(
        {
            "trajectories": all_recovery_trajectories,
            "history": recovery_history,
        },
        file,
    )

final_recovery_checkpoint = torch.load(
    best_recovery_checkpoint_path,
    map_location=DEVICE,
)

gru_model.load_state_dict(
    final_recovery_checkpoint["model_state_dict"]
)

print(
    "Loaded final checkpoint | "
    f"stage {final_recovery_checkpoint['stage']} | "
    f"round {final_recovery_checkpoint['round']} | "
    f"epoch {final_recovery_checkpoint['epoch']} | "
    f"success {final_recovery_checkpoint['success_rate']:.3f}"
)


Perturbed DAgger collection round 1:   0%|          | 0/400 [00:00<?, ?it/s]

Round 1 collection | trajectories 396 | mean length 14.21 | recovery disagreements 0.11
Fine-tune items | original: 1600 recovery: 1584


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

Perturbed DAgger round 1 | epoch 01 | loss 0.06268 | valid success 0.955 | avg steps 18.21
Saved new best recovery checkpoint: /kaggle/working/cnn_gru_perturbed_dagger_best.pt


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

...

Perturbed DAgger round 2 | epoch 01 | loss 0.04847 | valid success 0.945 | avg steps 19.21


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

Perturbed DAgger round 2 | epoch 02 | loss 0.04488 | valid success 0.940 | avg steps 19.73


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

Perturbed DAgger round 2 | epoch 03 | loss 0.04723 | valid success 0.945 | avg steps 19.23
Loaded final checkpoint | stage perturbed_dagger | round 1 | epoch 1 | success 0.955


In [39]:
# =============================================================================
# 6. Evaluación final y comparación directa con el baseline
# =============================================================================

import pandas as pd

final_recovery_eval = evaluate_gru_model(
    valid_scenarios
)

print("Baseline success:", BASELINE_SUCCESS)
print(
    "Perturbed DAgger success:",
    final_recovery_eval["success_rate"],
)
print("Baseline average steps:", BASELINE_AVG_STEPS)
print(
    "Perturbed DAgger average steps:",
    final_recovery_eval["avg_steps"],
)

if recovery_history:
    recovery_history_df = pd.DataFrame(
        recovery_history
    )
    display(recovery_history_df)

if final_recovery_eval["success_rate"] < BASELINE_SUCCESS:
    raise RuntimeError(
        "El rollback falló: el modelo final no debería ser peor "
        "que el baseline protegido."
    )


Evaluating CNN + GRU:   0%|          | 0/200 [00:00<?, ?it/s]

Baseline success: 0.945
Perturbed DAgger success: 0.955
Baseline average steps: 19.235
Perturbed DAgger average steps: 18.215


,round,epoch,loss,success_rate,avg_steps,recovery_trajectories
0,1,1,0.062678,0.955,18.215,396
1,1,2,0.067745,0.940,19.725,396
2,1,3,0.067289,0.945,19.145,396
3,2,1,0.048470,0.945,19.215,794
4,2,2,0.044881,0.940,19.725,794
5,2,3,0.047230,0.945,19.235,794


In [40]:
import json
import pandas as pd
from tqdm.auto import tqdm

submission_rows = []

for scenario in tqdm(
    test_scenarios,
    desc="Generating perturbed-DAgger submission",
):
    episode = run_episode_gru(scenario)

    action_ids = [
        int(action)
        for action in episode["actions"]
    ]

    submission_rows.append({
        "id": (
            f'{scenario["layout_id"]}'
            f'__{scenario["episode_seed"]}'
        ),
        "actions": json.dumps(
            action_ids,
            separators=(",", ":"),
        ),
    })

submission = pd.DataFrame(
    submission_rows,
    columns=["id", "actions"],
)

# Verificaciones antes de guardar
assert len(submission) == len(test_scenarios)
assert submission.columns.tolist() == ["id", "actions"]
assert submission["id"].is_unique
assert not submission.isna().any().any()

submission_path = OUTPUT_DIR / "submission_cnn_gru_perturbed_dagger.csv"
submission.to_csv(
    submission_path,
    index=False,
)

print("Saved:", submission_path)
print("Shape:", submission.shape)
print(submission.head())


Generating perturbed-DAgger submission:   0%|          | 0/1600 [00:00<?, ?it/s]

Saved: /kaggle/working/submission_cnn_gru_perturbed_dagger.csv
Shape: (1600, 2)
                  id                                      actions
0  test_0000__300000                        [1,2,2,0,4,1,1,3,3,5]
1  test_0000__300001          [2,2,2,1,4,3,3,3,3,3,0,1,1,3,3,0,5]
2  test_0000__300002                [0,0,4,0,0,2,2,2,0,2,2,0,0,5]
3  test_0000__300003  [3,3,3,3,1,1,1,1,3,1,1,3,3,0,4,0,0,0,0,0,5]
4  test_0001__300004                                  [0,3,4,1,5]
